In [4]:
#---BREAST CANCER 2--

import pandas as pd
import numpy as np


try:
    df = pd.read_csv("data_refined.csv")
except FileNotFoundError:
    print("Error: 'data_refined.csv' not found. Please run your preprocessing step first.")
    exit()

# Harmonize the target column name based on user specifications
# Detects whether the file uses 'Diagnosed' or 'diagnosis'
target_col = 'Diagnosed' if 'Diagnosed' in df.columns else 'diagnosis'

if target_col not in df.columns:
    print(f"Error: Target column ('Diagnosed' or 'diagnosis') not found in dataset columns: {df.columns.tolist()}")
    exit()

# Calculate absolute correlation coefficients with the target
# We use absolute value because strong negative correlations are just as predictive as strong positive ones
correlations = df.corr()[target_col].drop(target_col).abs()

# Set the correlation threshold limit
# A limit of 0.50 selects features with a strong linear relationship to tumor type
correlation_limit = 0.50

# Filter features that meet or exceed our threshold
important_features_series = correlations[correlations >= correlation_limit]

# Sort them in descending order of importance for better readability
important_features_series = important_features_series.sort_values(ascending=False)

# Extract the clean list of feature names
important_feature_names = important_features_series.index.tolist()

# Output the results
print(f"--- Correlation Analysis (Threshold Limit >= {correlation_limit}) ---")
print(f"Total features meeting the limit: {len(important_feature_names)} out of {len(correlations)}\n")

print("Selected Important Features & Their Absolute Correlation Coefficients:")
for feat, score in important_features_series.items():
    print(f"• {feat}: {score:.4f}")

print("\nFinal List of Important Feature Names for Training:")
print(important_feature_names)

#---STEP 3---
from sklearn.model_selection import train_test_split

# Load the preprocessed dataset
try:
    df = pd.read_csv("data_refined.csv")
except FileNotFoundError:
    print("Error: 'data_refined.csv' not found. Please run your preprocessing step first.")
    exit()

# Separate features (X) and target label (y)
target_col = 'Diagnosed' if 'Diagnosed' in df.columns else 'diagnosis'
X = df.drop(columns=[target_col])
y = df[target_col]

# First Split: Separate 10% of the entire dataset for the Test Set
# The remaining 90% goes into a temporary training/validation pool (X_temp, y_temp)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.10, random_state=42, stratify=y)

# Second Split: Separate the remaining 90% into Training and Validation sets
# To get exactly 10% of the original data for validation out of the 90% temporary pool:
# 0.10 / 0.90 = 1/9 ≈ 0.1111
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=(1/9), random_state=42, stratify=y_temp)

# Output and verify data dimensions
print("--- Data Splitting Complete ---")
print(f"Total original rows:     {len(df)}")
print(f"Training Set (80%):      {X_train.shape[0]} rows (Labels: {dict(y_train.value_counts())})")
print(f"Validation Set (10%):    {X_val.shape[0]} rows (Labels: {dict(y_val.value_counts())})")
print(f"Test Set (10%):          {X_test.shape[0]} rows (Labels: {dict(y_test.value_counts())})")

#---STEP 4---


from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix

# Load the preprocessed dataset
try:
    df = pd.read_csv("data_refined.csv")
except FileNotFoundError:
    print("Error: 'data_refined.csv' not found. Please verify your pipeline.")
    exit()

# Identify target and calculate feature subsets
target_col = 'Diagnosed' if 'Diagnosed' in df.columns else 'diagnosis'
X = df.drop(columns=[target_col])
y = df[target_col]

# Define the reduced feature set (Correlation limit >= 0.50)
correlations = df.corr()[target_col].drop(target_col).abs()
important_features = correlations[correlations >= 0.50].index.tolist()

# Divide data into 80% Train, 10% Val, 10% Test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.10, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=(1/9), random_state=42, stratify=y_temp)

# Isolate corresponding matrices for the reduced features
X_train_red = X_train[important_features]
X_val_red = X_val[important_features]
X_test_red = X_test[important_features]

# Optimize 'k' for KNN using 5-Fold Cross Validation on the training pool
k_range = {'n_neighbors': list(range(1, 21))}

grid_full = GridSearchCV(KNeighborsClassifier(), k_range, cv=5, scoring='accuracy')
grid_full.fit(X_train, y_train)
best_k_full = grid_full.best_params_['n_neighbors']

grid_red = GridSearchCV(KNeighborsClassifier(), k_range, cv=5, scoring='accuracy')
grid_red.fit(X_train_red, y_train)
best_k_red = grid_red.best_params_['n_neighbors']

# Initialize models
models = {
    'KNN (Full Features)': KNeighborsClassifier(n_neighbors=best_k_full),
    'KNN (Reduced Features)': KNeighborsClassifier(n_neighbors=best_k_red),
    'Random Forest (Full Features)': RandomForestClassifier(random_state=42),
    'Random Forest (Reduced Features)': RandomForestClassifier(random_state=42),
    'SVC (Full Features)': SVC(random_state=42),
    'SVC (Reduced Features)': SVC(random_state=42)
}

# Train and evaluate on the Holdout Test Set
print("--- MODEL PERFORMANCE EVALUATION RESULTS ---")
print(f"Optimal K found (Full Set):    k = {best_k_full}")
print(f"Optimal K found (Reduced Set): k = {best_k_red}\n")

performance_summary = []

for name, model in models.items():
    is_reduced = 'Reduced' in name
    # Pick appropriate data matrix split
    X_tr = X_train_red if is_reduced else X_train
    X_te = X_test_red if is_reduced else X_test
    
    # Train
    model.fit(X_tr, y_train)
    # Predict & Score
    predictions = model.predict(X_te)
    accuracy = accuracy_score(y_test, predictions)
    matrix = confusion_matrix(y_test, predictions)
    
    status = "PASSED (>=94%)" if accuracy >= 0.94 else "FAILED (<94%)"
    print(f"================ {name} ================")
    print(f"Accuracy Score: {accuracy * 100:.2f}% -> {status}")
    print("Confusion Matrix:")
    print(matrix)
    print()

#---STEP 5---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix

# Load the preprocessed dataset
try:
    df = pd.read_csv("data_refined.csv")
except FileNotFoundError:
    print("Error: 'data_refined.csv' not found. Please run your preprocessing step first.")
    exit()

# Separate features (X) and target label (y)
target_col = 'Diagnosed' if 'Diagnosed' in df.columns else 'diagnosis'
X = df.drop(columns=[target_col])
y = df[target_col]

# Divide data into 80% Train, 10% Val, 10% Test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.10, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=(1/9), random_state=42, stratify=y_temp)

# Apply PCA Dimensionality Reduction
# n_components=0.95 tells scikit-learn to pick the minimum number of components needed to retain 95% variance
pca = PCA(n_components=0.95, random_state=42)

# Fit PCA ONLY on the training data to prevent data leakage
X_train_pca = pca.fit_transform(X_train)
X_val_pca = pca.transform(X_val)
X_test_pca = pca.transform(X_test)

num_components = pca.n_components_
print("--- PCA Feature Reduction Summary ---")
print(f"Original feature dimensions: {X_train.shape[1]}")
print(f"Reduced PCA components retaining 95% variance: {num_components}\n")

# Find the optimal 'k' for KNN on the new PCA space using 5-Fold Cross Validation
k_range = {'n_neighbors': list(range(1, 21))}
grid_knn = GridSearchCV(KNeighborsClassifier(), k_range, cv=5, scoring='accuracy')
grid_knn.fit(X_train_pca, y_train)
best_k_pca = grid_knn.best_params_['n_neighbors']

# Initialize models using the new PCA feature space
models_pca = {
    f'KNN (PCA, k={best_k_pca})': KNeighborsClassifier(n_neighbors=best_k_pca),
    'Random Forest (PCA)': RandomForestClassifier(random_state=42),
    'SVC (PCA)': SVC(random_state=42)
}

# Train and Evaluate Classifiers on Holdout Test Set
print("--- PCA CLASS-MODEL PERFORMANCE EVALUATION ---")
for name, model in models_pca.items():
    # Fit model to PCA dimensions
    model.fit(X_train_pca, y_train)
    
    # Predict and calculate metrics
    predictions = model.predict(X_test_pca)
    accuracy = accuracy_score(y_test, predictions)
    matrix = confusion_matrix(y_test, predictions)
    
    # Check minimum accuracy criteria (94%)
    status = "PASSED (>=94%)" if accuracy >= 0.94 else "FAILED (<94%)"
    
    print(f"================ {name} ================")
    print(f"Accuracy Score: {accuracy * 100:.2f}% -> {status}")
    print("Confusion Matrix:")
    print(matrix)
    print()





--- Correlation Analysis (Threshold Limit >= 0.5) ---
Total features meeting the limit: 15 out of 30

Selected Important Features & Their Absolute Correlation Coefficients:
• concave points_worst: 0.7936
• perimeter_worst: 0.7829
• concave points_mean: 0.7766
• radius_worst: 0.7765
• perimeter_mean: 0.7426
• area_worst: 0.7338
• radius_mean: 0.7300
• area_mean: 0.7090
• concavity_mean: 0.6964
• concavity_worst: 0.6596
• compactness_mean: 0.5965
• compactness_worst: 0.5910
• radius_se: 0.5671
• perimeter_se: 0.5561
• area_se: 0.5482

Final List of Important Feature Names for Training:
['concave points_worst', 'perimeter_worst', 'concave points_mean', 'radius_worst', 'perimeter_mean', 'area_worst', 'radius_mean', 'area_mean', 'concavity_mean', 'concavity_worst', 'compactness_mean', 'compactness_worst', 'radius_se', 'perimeter_se', 'area_se']
--- Data Splitting Complete ---
Total original rows:     569
Training Set (80%):      455 rows (Labels: {0: np.int64(285), 1: np.int64(170)})
Valida